In [18]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

In [19]:
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports/json', exist_ok=True)
os.makedirs('../reports/html', exist_ok=True)

In [20]:
# 1. Load the model directly
model_path = '../models/baselines/lightgbm_Sleep_Quality_Num.joblib'
lgbm_model = joblib.load(model_path)

/media/nata/7AAE88B3AE886989/Навчання/4 курс/Diploma/DS-Research-NS/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:
# 2. Load the data
df = pd.read_csv('../data/processed/features_engineered_with_clusters.csv')

In [22]:
# Drop the targets so they don't leak into the features
targets = ['Sleep_Quality', 'Stress_Level', 'Health_Issues']
X_raw = df.drop(columns=targets, errors='ignore')

In [ ]:
# 3. One-hot encode the categorical variable
X_encoded = pd.get_dummies(X_raw)

In [24]:
temp_model = joblib.load('../models/baselines/lightgbm_Sleep_Quality_Num.joblib')
expected_features = temp_model.feature_name_

for col in expected_features:
    if col not in X_encoded.columns:
        X_encoded[col] = 0
X_final = X_encoded[expected_features]
X_sample = X_final.sample(n=1000, random_state=42)

/media/nata/7AAE88B3AE886989/Навчання/4 курс/Diploma/DS-Research-NS/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
for target in targets:
    model_path = f'../models/baselines/lightgbm_{target}_Num.joblib'
    
    if not os.path.exists(model_path):
        print(f"Model not found: {model_path}. Skipping.")
        continue
        
    print(f"Analyzing {target}")
    model = joblib.load(model_path)
    
    # Run SHAP
    explainer = shap.TreeExplainer(model)
    shap_explanation = explainer(X_sample)
    shap_values_array = shap_explanation.values
    
    # Save Summary Plots
    plt.figure(figsize=(10, 6))
    if len(shap_values_array.shape) == 3:
        shap.summary_plot(shap_values_array, X_sample, plot_type="bar", show=False)
        plt.savefig(f'../reports/figures/shap_bar_{target}.png', bbox_inches='tight')
        plt.close()
        
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values_array[:, :, 1], X_sample, show=False)
        plt.savefig(f'../reports/figures/shap_beeswarm_{target}.png', bbox_inches='tight')
        plt.close()
    else:
        # Binary/Regression
        shap.summary_plot(shap_values_array, X_sample, plot_type="bar", show=False)
        plt.savefig(f'../reports/figures/shap_bar_{target}.png', bbox_inches='tight')
        plt.close()
        
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values_array, X_sample, show=False)
        plt.savefig(f'../reports/figures/shap_beeswarm_{target}.png', bbox_inches='tight')
        plt.close()

    user_index = 0
    if len(shap_explanation.values.shape) == 3:
        user_shap_values = shap_explanation.values[user_index, :, 1]
        base_value = float(shap_explanation.base_values[user_index, 1])
    else:
        user_shap_values = shap_explanation.values[user_index]
        base_value = float(shap_explanation.base_values[user_index])
    user_data = shap_explanation.data[user_index]
    
    # Save HTML Force Plot Snippet
    force_plot = shap.force_plot(
        base_value, 
        user_shap_values, 
        user_data, 
        feature_names=X_sample.columns.tolist()
    )
    shap.save_html(f'../reports/html/shap_force_plot_{target}.html', force_plot)

    # Construct and Save JSON Payload 
    feature_impacts = []
    all_shap_sum = 0.0

    for i, feature_name in enumerate(X_sample.columns):
        shap_val = float(user_shap_values[i])
        feat_val = float(user_data[i])
        all_shap_sum += shap_val
        
        if abs(shap_val) < 0.001:
            continue
            
        feature_impacts.append({
            "feature": feature_name,
            "value": round(feat_val, 2),
            "shap_influence": round(shap_val, 4)
        })

    feature_impacts.sort(key=lambda x: abs(x["shap_influence"]), reverse=True)
    
    shap_json_payload = {
        "prediction_id": f"poc-sample-{target}-123",
        "target_model": target,
        "base_value": round(base_value, 4),
        "final_prediction": round(base_value + all_shap_sum, 4),
        "top_drivers": feature_impacts[:5]
    }

    with open(f'../reports/json/shap_payload_{target}.json', 'w') as f:
        json.dump(shap_json_payload, f, indent=4)


/media/nata/7AAE88B3AE886989/Навчання/4 курс/Diploma/DS-Research-NS/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Analyzing Sleep_Quality
Analyzing Stress_Level


/media/nata/7AAE88B3AE886989/Навчання/4 курс/Diploma/DS-Research-NS/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Analyzing Health_Issues


/media/nata/7AAE88B3AE886989/Навчання/4 курс/Diploma/DS-Research-NS/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
